##Data Definitions

In [0]:
CATALOG_NAME = "ml_training_dev"
SOURCE_SCHEMA_NAME = "feature"
TARGET_SCHEMA_NAME = "training"
#TARGET_TABLE_NAME = "customers"

In [0]:
# label = 1 → cliente comprou o produto
# label = 0 → cliente não comprou o produto

In [0]:
# TRAIN: janeiro/2017 → setembro/2018
# TEST:  outubro/2018 → dezembro/2018

##Import Libraries

In [0]:
from pyspark.sql.functions import lit, count, row_number, rand, col
from pyspark.sql.window import Window

##Reading Source Tables

In [0]:
%sql
-- select max(last_purchase) last_purchase
-- from ml_training_dev.feature.customer_product_interactions
-- ---where last_purchase between "2018-08-01" and "2018-08-28"

In [0]:
customer_features = spark.table("ml_training_dev.feature.customers")
product_features = spark.table("ml_training_dev.feature.products")  
interaction_features = spark.table("ml_training_dev.feature.customer_product_interactions")

##Transformations

In [0]:
product_features = product_features.withColumnRenamed("total_orders", "total_product_orders")
                    
interaction_features = (interaction_features
                         .withColumnRenamed("total_spent", "interaction_total_spent")
                         .withColumnRenamed("avg_price", "interaction_avg_price")
                         .withColumnRenamed("avg_freight", "interaction_avg_freight")
                         )

###Filtering Distinct Data

In [0]:
customers = (
    customer_features
    .select("customer_unique_id")
    .distinct()
)

products = (
    product_features
    .select("product_id")
    .distinct()
)

###Labeling data

####Positive

In [0]:
positive = (
    interaction_features
    .select(
        "customer_unique_id",
        "product_id"
    ).distinct()
    .withColumn("label",lit(1))
)

In [0]:
candidate_pairs = customers.crossJoin(products)

####Positive

In [0]:
negative_candidates = (
    candidate_pairs
    .join(
        positive.select(
            "customer_unique_id",
            "product_id"
        ),
        on=["customer_unique_id", "product_id"],
        how="left_anti"
    )
)

In [0]:
positive_count = (
    positive
    .groupBy("customer_unique_id")
    .agg(
        count("*").alias("positive_count")
    )
)

In [0]:
negative_candidates = (
    negative_candidates
    .join(
        positive_count,
        on="customer_unique_id",
        how="inner"
    )
)

In [0]:
#negative_candidates.select('positive_count').distinct().display()

In [0]:
window = Window.partitionBy(
    "customer_unique_id"
).orderBy(rand(seed=42))

negative = (
    negative_candidates
    .withColumn("rn", row_number().over(window))
    .filter(
        col("rn") <= col("positive_count") * 3
    )
    .select(
        "customer_unique_id",
        "product_id"
    )
    .withColumn("label", lit(0))
)

In [0]:
recommendation_training = (
    interaction_features
    .join(
        customer_features,
        on="customer_unique_id",
        how="left"
    )
    .join(
        product_features,
        on="product_id",
        how="left"
    ))

In [0]:
feature_cols = [
    "total_orders",
    "total_items",
    "total_spent",
    "avg_order_value",
    "unique_products",
    "unique_categories",

    "avg_price",
    "avg_freight_value",
    "total_product_orders",
    "unique_customers",
    "total_revenue",
    "avg_rating",

    "purchases",
    "interaction_total_spent",
    "interaction_avg_price",
    "interaction_avg_freight"
]

In [0]:
training = training.fillna(0, subset=feature_cols)#.withColumn("label",lit(1.0))

In [0]:
training.printSchema()

In [0]:
training.select('purchases').distinct().display()

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

train_vector = assembler.transform(training)
train_vector.display()

In [0]:
from pyspark.ml.classification import LogisticRegression
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability"
)

model = lr.fit(train_vector)

In [0]:
from pyspark.sql.functions import lit

training = (
    training
    .fillna(0, subset=feature_cols)
    .withColumn("label", lit(1.0))
)

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

train_vector = assembler.transform(training)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability"
)

model = lr.fit(train_vector)

In [0]:
predictions = model.transform(test_vector)

In [0]:
# P(buy product | customer)

In [0]:
recommendations = (
    predictions
    .withColumn(
        "probability_buy",
        F.col("probability")[1]
    )
    .withColumn(
        "rank",
        F.row_number().over(window_customer)
    )
    .filter(F.col("rank") <= 10)
)